# PySpark DataFrame Practice Notebook

In this notebook, we will learn:

- How to read data into a DataFrame
- Basic DataFrame operations
- Filtering and transformations
- Aggregations
- Joins
- Window functions
- Writing data

This simulates a real-world data engineering workflow.

## Step 1: Initialize Spark Session

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark Practice") \
    .getOrCreate()

spark

## Step 2: Read Data into DataFrame

We will load a file into a PySpark DataFrame.

In [0]:
df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("path/to/your/file.csv")

df.show(5)

## Step 3: Explore Data

Let’s understand the structure and schema.

In [0]:
df.printSchema()
df.columns
df.describe().show()
df.count()

## Step 4: Select Columns

In [0]:
df.select("column1", "column2").show()

## Step 5: Filter Data

In [0]:
df.filter(df["column1"] > 100).show()

# Multiple conditions
from pyspark.sql.functions import col

df.filter((col("column1") > 100) & (col("column2") == "A")).show()

## Step 6: Add or Modify Columns

In [0]:
from pyspark.sql.functions import lit

df = df.withColumn("new_column", lit("test"))

df.show()

## Step 7: Rename Columns

In [0]:
df = df.withColumnRenamed("column1", "new_column1")

## Step 8: Drop Columns

In [0]:
df = df.drop("unwanted_column")

## Step 9: Handle Missing Values

In [0]:
df.na.drop().show()

df.na.fill({"column1": 0}).show()

## Step 10: Remove Duplicate Records

In [0]:
df.dropDuplicates().show()

## Step 11: Aggregations

In [0]:
from pyspark.sql.functions import sum, avg, count

df.groupBy("column2").agg(
    sum("column1").alias("total"),
    avg("column1").alias("average"),
    count("*").alias("count")
).show()

## Step 12: Sorting Data

In [0]:
df.orderBy("column1", ascending=False).show()

## Step 13: Joins

We will join two DataFrames.

In [0]:
df2 = spark.read.csv("path/to/another/file.csv", header=True, inferSchema=True)

joined_df = df.join(df2, on="id", how="inner")

joined_df.show()

## Step 14: Window Functions

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("column2").orderBy("column1")

df = df.withColumn("rank", row_number().over(window_spec))

df.show()

## Step 15: Conditional Columns

In [0]:
from pyspark.sql.functions import when

df = df.withColumn(
    "category",
    when(col("column1") > 100, "High")
    .otherwise("Low")
)

df.show()

## Step 16: Date Functions

In [0]:
from pyspark.sql.functions import to_date, year, month

df = df.withColumn("date", to_date(col("date_column"), "yyyy-MM-dd"))

df = df.withColumn("year", year("date")) \
       .withColumn("month", month("date"))

df.show()

## Step 17: Cache DataFrame

In [0]:
df.cache()
df.count()

## Step 18: Write Data to Storage

In [0]:
df.write \
    .mode("overwrite") \
    .format("parquet") \
    .save("path/to/output")

## Step 19: Using Spark SQL

In [0]:
df.createOrReplaceTempView("table")

spark.sql("""
SELECT column2, COUNT(*) as total
FROM table
GROUP BY column2
""").show()

## Key Learnings

- PySpark is lazy → transformations are not executed until an action
- Always use column functions instead of Python logic
- Prefer DataFrame API over RDDs
- Optimize using:
  - partitioning
  - caching
  - predicate pushdown